# 02.4 – Audit and repair of the resort tables

**Owner:** Luca (repair, requested by the group on 28 Nov 2026) · **Original owner of the data:** Lorenzo (notebooks 02.2 and 02.3)

This notebook checks the resort tables produced by notebooks 02.2 / 02.3 and rebuilds `resorts_clean.csv` and `resorts_features.csv` **without inventing any value**.

What was wrong (all checks are run below, so the evidence is reproducible):

| # | Problem found | Repair |
| --- | --- | --- |
| 1 | `hotels_4_5_star_10km` was not measured. All 300 values equal `round(slope_km/30 + price/25 − 2)`: a formula of the price (the variable we want to explain). | Column set to missing. Real counts come from `src/fetch_hotels_overpass.py`. |
| 2 | `slope_km` often holds the length of the **ski routes** instead of the **slopes** (Ischgl 15 km instead of 239 km, Sölden 8.4 instead of 144.1, Zermatt 38 instead of 322). | Corrected parser `src/skiresort_parser.py`; `src/reparse_skiresort.py` re-reads the cached pages. Until it is run, the column is flagged `unverified_old_parser`. |
| 3 | 12 resorts were added by hand to reach exactly 75 per country. All their numbers are the national median (Ischgl, Westendorf and Ellmau are identical). | Removed. Their names are listed in the audit file. |
| 4 | Two rows carry the URL of one resort but the hand-typed numbers of another ("Scuol" with the Ischgl URL, "Pila" with the Espace San Bernardo URL). Five real resorts were dropped by the quota. | Rebuilt from the original scraped table, one row per Skiresort URL. |
| 5 | `lifts_count`: the 150 values are all 12, a side effect of two hand-typed rows and the country median. `snow_cannons` is 0 for every resort (a parsing failure). | Both cleared; lifts come back through the corrected parser. `snow_cannons` is dropped. |
| 6 | Coordinates were merged by resort name; the names contain invisible characters, so 291 of 300 rows lost latitude, longitude and distances. | Coordinates joined by URL from `resorts_snow_roster.csv` (Matthias, 33 checked on the source page), compared with the second geocoding run and screened for isolated points; **20 wrong positions corrected by hand** (approximate, flagged); distances recomputed. `coordinate_check` tells how reliable each point is. The full independent check is `python src/verify_resort_coordinates.py`. |

The rule for the whole notebook: **missing is better than invented.**

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent
PROCESSED = ROOT / "data" / "processed"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

def slug(url):
    return str(url).rstrip("/").split("/")[-1]

# Inputs (all committed in the repository)
scraped = pd.read_csv(PROCESSED / "resorts_scraped_before_quota.csv")   # table before the quota step (commit 5625dec)
lorenzo = pd.read_csv(PROCESSED / "resorts_features_25nov_before_repair.csv")   # table as pushed on 25 Nov 2026 (58097fe)
roster  = pd.read_csv(PROCESSED / "resorts_snow_roster.csv")            # one row per ski area (Matthias)
scraped["resort_id"] = scraped["url"].map(slug)
lorenzo["resort_id"] = lorenzo["url"].map(slug)
print("scraped table:", scraped.shape, "| table of 25 Nov:", lorenzo.shape, "| roster:", roster.shape)

scraped table: (300, 23) | table of 25 Nov: (300, 21) | roster: (293, 25)


## 1. Audit: show the problems with code

In [2]:
# 1) Hotel counts versus the formula
formula = np.clip(np.round(scraped["slope_km"] / 30 + scraped["price_day_eur"] / 25 - 2), 0, None).astype(int)
same = int((formula == scraped["hotels_4_5_star_10km"]).sum())
print(f"1) Hotel counts equal to the formula slope/30 + price/25 - 2: {same} of {len(scraped)} rows")

# 3) Hand-added rows in the table of 25 Nov
manual = lorenzo[lorenzo["manually_added"] == True]
print(f"3) Rows with manually_added = True: {len(manual)}; distinct (elevation_top, slope, price) combinations: "
      f"{manual[['elevation_top_m','slope_km','price_day_eur']].drop_duplicates().shape[0]}")
print("   ", ", ".join(manual["resort_name"]))

# 4) Rows whose URL belongs to another resort
wrong = lorenzo[lorenzo["resort_name"].isin(["Scuol – Motta Naluns", "Pila – Aosta"])][["resort_name", "url"]]
print("4) Names with the URL of another resort:")
print(wrong.to_string(index=False))
lost = scraped[~scraped["resort_id"].isin(lorenzo["resort_id"])].drop_duplicates("resort_id")
print("   real resorts missing in the table of 25 Nov:", ", ".join(lost["resort_id"]))

# 5) lifts_count and snow_cannons
print("5) lifts_count values in the table of 25 Nov:", lorenzo["lifts_count"].value_counts(dropna=False).to_dict())
print("   snow_cannons values in the original table:", scraped["snow_cannons"].value_counts().to_dict())

# 6) Coordinates in the table of 25 Nov
print("6) Rows without latitude in the table of 25 Nov:", int(lorenzo["latitude"].isna().sum()), "of", len(lorenzo))

1) Hotel counts equal to the formula slope/30 + price/25 - 2: 300 of 300 rows
3) Rows with manually_added = True: 12; distinct (elevation_top, slope, price) combinations: 6
    Bergün, Sauze d'Oulx, Ischgl, Westendorf, La Rosière, Le Grand-Bornand, Kandersteg, Val di Fassa, Ellmau, Les Sept Laux, Autrans, Super Besse
4) Names with the URL of another resort:
         resort_name                                                                               url
Scuol – Motta Naluns           https://www.skiresort.com/en/ski-resort/ischgl-samnaun-silvretta-arena/
        Pila – Aosta https://www.skiresort.com/en/ski-resort/espace-san-bernardo-la-rosiere-la-thuile/
   real resorts missing in the table of 25 Nov: scuol-motta-naluns, pila, axamer-lizum, see, sainte-foy-tarentaise
5) lifts_count values in the table of 25 Nov: {12.0: 150, nan: 150}
   snow_cannons values in the original table: {0: 300}
6) Rows without latitude in the table of 25 Nov: 291 of 300


In [3]:
# 2) slope_km: compare with the public Skiresort pages (read on 28 Nov 2026).
# The page says: "there are <slopes> km of slopes and <routes> km of ski routes available".
PAGE_CHECK = pd.DataFrame([
    {"resort_id": "ischgl-samnaun-silvretta-arena", "slopes_km_page": 239.0, "ski_routes_km_page": 15.0},
    {"resort_id": "soelden", "slopes_km_page": 144.1, "ski_routes_km_page": 8.4},
    {"resort_id": "zermatt-breuil-cervinia-valtournenche-matterhorn", "slopes_km_page": 322.0, "ski_routes_km_page": 38.0},
])
check = PAGE_CHECK.merge(scraped.drop_duplicates("resort_id")[["resort_id", "slope_km"]], on="resort_id", how="left")
check["our_value_equals_ski_routes"] = check["slope_km"] == check["ski_routes_km_page"]
check

,resort_id,slopes_km_page,ski_routes_km_page,slope_km,our_value_equals_ski_routes
0,ischgl-samnaun-silvretta-arena,239.0,15.0,15.0,True
1,soelden,144.1,8.4,8.4,True
2,zermatt-breuil-cervinia-valtournenche-matterhorn,322.0,38.0,38.0,True


In all three cases the value in our table is the **ski-routes** figure, not the slope length. The old pattern took the first "km" after the word *slopes*, which on these pages is the ski-routes number. Because it affects every resort whose page lists ski routes, `slope_km` cannot be trusted until the pages are read again with the corrected parser.

## 2. Repair

### 2.1 One row per ski area
The original scraped table has 300 rows but only 293 distinct Skiresort URLs (the same area is listed in two countries, or twice in one country). For cross-border areas we keep the row of one **canonical country**; the other member country stays in `country_memberships`.

The canonical country is our own documented decision (country of the main ticket office / price shown on the page).

In [4]:
CANONICAL_COUNTRY = {
    "les-portes-du-soleil-morzine-avoriaz-les-gets-chatel-morgins-champery": "FR",
    "via-lattea-sestriere-sauze-doulx-san-sicario-claviere-montgenevre": "IT",
    "zermatt-breuil-cervinia-valtournenche-matterhorn": "CH",
    "ischgl-samnaun-silvretta-arena": "AT",
    "espace-san-bernardo-la-rosiere-la-thuile": "IT",
}

df = scraped.copy()
df["resort_name"] = df["resort_name"].str.replace("\u200b", "", regex=False).str.strip()
df["name_length"] = df["resort_name"].str.len()          # the repeated rows with a very long name contain page text
df["is_canonical"] = df.apply(lambda r: CANONICAL_COUNTRY.get(r["resort_id"], r["country"]) == r["country"], axis=1)
df = (df.sort_values(["resort_id", "is_canonical", "name_length"], ascending=[True, False, True])
        .drop_duplicates("resort_id", keep="first")
        .drop(columns=["name_length", "is_canonical"]))
integration = {"rows_before": len(scraped), "rows_after": len(df), "rows_removed_as_duplicates": len(scraped) - len(df)}
print(integration)
print(df["country"].value_counts().to_dict())

{'rows_before': 300, 'rows_after': 293, 'rows_removed_as_duplicates': 7}
{'AT': 75, 'IT': 74, 'CH': 73, 'FR': 71}


### 2.2 Roster fields, coordinates and distances
Coordinates come from `resorts_snow_roster.csv` (joined by `resort_id`, not by name): 33 areas were checked against the first lift access point on the Skiresort page (`verified_access_point`), the other 260 keep the original geocode.

The geocoding run of 25 Nov is used as a second opinion, but it is **not independent**: both runs search the resort *name* and can make the same mistake. Agreement within 5 km is therefore called `geocoders_agree_not_independent` and is *not* proof of a correct position. Three screens were applied:

1. **Second opinion** (differences above 5 km are flagged `two_geocodes_differ_review`).
2. **Isolation screen:** an Alpine resort that is more than 25 km away from every other resort in the table is suspicious (the Alps are densely covered). This found eight positions that are hundreds of kilometres wrong although both geocoders agreed (for example Nara–Leontica was placed in Zurich and St. Anton am Arlberg in Lower Austria).
3. **Manual review of all 260 unchecked positions** against the geography of the place names. This found twelve further wrong positions inside the Alps.

All corrections are written in `COORDINATE_FIXES` below with the reason, are approximate (a few km) and are marked `manual_correction_approximate`. The independent check for all 293 areas (first lift-access point of the Skiresort page, as Matthias did for 33) is `python src/verify_resort_coordinates.py`, to be run once in the Codespace.

In [5]:
roster_cols = ["resort_id", "country_memberships", "latitude", "longitude", "coordinate_method",
               "scope_decision", "include_in_alpine_candidate_set"]
df = df.drop(columns=["latitude", "longitude"]).merge(roster[roster_cols], on="resort_id", how="left", validate="1:1")
assert df["latitude"].notna().all(), "every resort must be in the roster"
df = df.rename(columns={"include_in_alpine_candidate_set": "in_alpine_candidate_set"})
df["in_alpine_candidate_set"] = df["in_alpine_candidate_set"].astype(bool)
df["cross_border"] = df["country_memberships"].str.contains(";")

# Second opinion: the independent geocoding run of 25 Nov (resorts_geocoded_cache.csv, joined by cleaned name)
cache = pd.read_csv(PROCESSED / "resorts_geocoded_cache.csv")
cache["key"] = cache["resort_name"].str.replace("​", "", regex=False).str.strip()
cache = cache.drop_duplicates("key").set_index("key")
df["cache_latitude"] = df["resort_name"].map(cache["latitude"])
df["cache_longitude"] = df["resort_name"].map(cache["longitude"])

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))

df["geocode_difference_km"] = haversine_km(df["latitude"], df["longitude"], df["cache_latitude"], df["cache_longitude"]).round(1)
df["coordinate_check"] = np.select(
    [df["coordinate_method"] == "source_first_listed_lift_access",
     df["geocode_difference_km"].isna(),
     df["geocode_difference_km"] <= 5],
    ["verified_access_point", "single_geocode_only", "geocoders_agree_not_independent"],
    default="two_geocodes_differ_review")
print(df["coordinate_check"].value_counts().to_dict())
print(df.loc[df["coordinate_check"] == "two_geocodes_differ_review",
             ["resort_id", "latitude", "longitude", "cache_latitude", "cache_longitude", "geocode_difference_km"]].to_string(index=False))

# Documented corrections. In the review above the first geocode is right for Sangiacomo, Vars, Les Arcs and Villars
# (the second one puts them in Veneto, Burgundy, Provence and Fribourg); for Rosskopf (Sterzing) the second one is right.
COORDINATE_FIXES = {
    "sainte-foy-tarentaise": (45.59, 6.89, "geocode was near Saint-Etienne (4.39 E); village is in Tarentaise, approximate value, verify"),
    "rosskopf-monte-cavallo-sterzing-vipiteno": (46.914, 11.384, "first geocode 51 km away in the Passeier valley; second geocode is at Sterzing"),
    # Found by the isolation screen (both geocoders agreed on a wrong place)
    "nara-leontica": (46.46, 8.92, "geocoded in Zurich (47.38 N); the ski area is above Leontica, Blenio valley (Ticino); snow-forecast.com gives 46.46 N 8.92 E"),
    "st-anton-st-christoph-stuben-lech-zuers-warth-schroecken-ski-arlberg": (47.13, 10.27, "geocoded in Lower Austria (15.2 E); St. Anton am Arlberg is at 47.13 N 10.27 E"),
    "hoch-imst-imst": (47.24, 10.73, "geocoded in Carinthia (14.95 E); Imst is in Tyrol at 47.24 N 10.74 E"),
    "belpiano-schoeneben-malga-san-valentino-haideralm": (46.80, 10.51, "geocoded near Genoa (44.4 N); Schoeneben/Haideralm is at the Reschen pass, South Tyrol"),
    "balme-les-autannes-vallorcine-le-tour": (46.03, 6.93, "geocoded in the Jura plain (5.38 E); Vallorcine is near Chamonix"),
    "les-portes-du-mont-blanc-combloux-megeve-le-jaillet-la-giettaz": (45.86, 6.62, "geocoded in Provence (43.87 N); Megeve is at 45.86 N 6.62 E"),
    "saint-lary-soulan": (42.82, 0.32, "geocoded near Nice (7.28 E); Saint-Lary-Soulan is in the Pyrenees"),
    "font-romeu-bolquere-pyrenees-2000": (42.50, 2.03, "geocoded in Switzerland (6.82 E); Font-Romeu is in the Pyrenees"),
    # Found by the manual review of the 260 unchecked positions
    "belvedere-col-rodella": (46.48, 11.77, "geocoded in Piedmont (7.66 E); Canazei is in the Dolomites (Val di Fassa)"),
    "pila": (45.70, 7.31, "geocoded in Piedmont (8.08 E); Pila is above Aosta"),
    "cimone-montecreto": (44.23, 10.77, "geocoded near Trento (45.98 N); Monte Cimone / Sestola is in the Emilian Apennines"),
    "gurgl-obergurgl-hochgurgl": (46.87, 11.03, "geocoded near Seefeld (47.36 N); Obergurgl/Hochgurgl is in the Oetztal at 46.87 N 11.03 E"),
    "nassfeld-hermagor": (46.57, 13.28, "geocoded near Bad Gastein (47.06 N); Nassfeld is on the Italian border at 46.57 N 13.28 E"),
    "eggli-la-videmanette": (46.47, 7.28, "geocoded in Glarus (8.98 E); the ski area is at Gstaad/Saanen"),
    "saas-fee": (46.11, 7.93, "geocoded in the Praettigau (9.81 E, Saas im Praettigau); Saas-Fee is in the Valais"),
    "first-grindelwald": (46.62, 8.04, "geocoded near Kandersteg (7.64 E); Grindelwald is at 46.62 N 8.04 E"),
    "lagorai-passo-brocon": (46.06, 11.64, "geocoded next to Alpe Cermis; Passo Brocon / Castello Tesino is 25 km south-east"),
    "valchiavenna-madesimo-campodolcino": (46.43, 9.35, "geocoded near Lake Como (46.23 N); Madesimo/Campodolcino are in the upper Valchiavenna"),
}
for _rid in COORDINATE_FIXES:
    assert (df["resort_id"].str.startswith(_rid)).sum() == 1, ("unknown or ambiguous resort_id", _rid)
COORDINATE_FIXES = {df.loc[df["resort_id"].str.startswith(k), "resort_id"].iloc[0]: v for k, v in COORDINATE_FIXES.items()}
for rid, (lat, lon, reason) in COORDINATE_FIXES.items():
    m = df["resort_id"] == rid
    df.loc[m, ["latitude", "longitude"]] = (lat, lon)
    df.loc[m, "coordinate_method"] = "manual_correction_approximate"
    df.loc[m, "coordinate_check"] = "manual_correction"

# Independent verification against the Skiresort page: python src/verify_resort_coordinates.py (Codespace).
# Used only when the file is complete (all resorts); a partial file is ignored, never half-applied.
VERIFICATION_ACTIONS = []
verification_file = PROCESSED / "resort_coordinate_verification.csv"
if verification_file.exists():
    ver = pd.read_csv(verification_file).drop_duplicates("resort_id").set_index("resort_id")
    if set(ver.index) == set(df["resort_id"]):
        for i, r in df.iterrows():
            v = ver.loc[r["resort_id"]]
            if v["status"] == "mismatch":
                VERIFICATION_ACTIONS.append((r["resort_id"], "replaced", f"stored position {v['nearest_access_km']} km from the nearest access point; "
                                             f"replaced by the first listed lift-access point ({v['first_access_latitude']:.5f}, {v['first_access_longitude']:.5f})"))
                df.loc[i, ["latitude", "longitude"]] = (v["first_access_latitude"], v["first_access_longitude"])
                df.loc[i, "coordinate_method"] = "source_first_listed_lift_access"
                df.loc[i, "coordinate_check"] = "verified_access_point"
            elif v["status"] == "confirmed":
                df.loc[i, "coordinate_check"] = "verified_access_point"
            else:
                df.loc[i, "coordinate_check"] = "access_point_within_15km_review"
            if v["country_match"] is False or str(v["country_match"]) == "False":
                VERIFICATION_ACTIONS.append((r["resort_id"], "flagged", f"the source page lists {v['source_country']}, the table says {r['country_memberships']}; decide the canonical country"))
        print("Verification applied:", pd.Series([a[1] for a in VERIFICATION_ACTIONS]).value_counts().to_dict(), "|", df["coordinate_check"].value_counts().to_dict())
    else:
        print("resort_coordinate_verification.csv is incomplete (", len(ver), "of", len(df), "resorts): ignored. Run src/verify_resort_coordinates.py again.")
else:
    print("resort_coordinate_verification.csv not found: coordinates keep the checks above (run src/verify_resort_coordinates.py in the Codespace).")

# Scope review flags. Matthias's scope decision (275 candidates) is not changed, because his pipeline depends on it.
# These areas are outside the Alps by geography; the analysis set below leaves them out.
SCOPE_REVIEW = {
    "hochficht": "Hochficht (Upper Austria, 48.7 N) is in the Bohemian Forest, not in the Alps",
    "abetone-val-di-luce": "Abetone is in the Tuscan-Emilian Apennines, not in the Alps (same rule as the 3 Apennine areas already excluded)",
    "lelex-crozet": "Lelex-Crozet is in the Jura, not in the Alps",
    "saint-lary-soulan": "Saint-Lary-Soulan is in the Pyrenees, not in the Alps (same rule as the 13 Pyrenees areas already excluded)",
    "font-romeu-bolquere-pyrenees-2000": "Font-Romeu is in the Pyrenees, not in the Alps (same rule as the 13 Pyrenees areas already excluded)",
}
SCOPE_REVIEW = {df.loc[df["resort_id"].str.startswith(k), "resort_id"].iloc[0]: v for k, v in SCOPE_REVIEW.items()}
SCOPE_REVIEW[df.loc[df["resort_id"].str.startswith("cimone-montecreto"), "resort_id"].iloc[0]] = "Monte Cimone/Sestola is in the Emilian Apennines, not in the Alps"
df["alps_review_flag"] = ""
for rid, reason in SCOPE_REVIEW.items():
    df.loc[df["resort_id"] == rid, "alps_review_flag"] = reason
df["in_alpine_analysis_set"] = df["in_alpine_candidate_set"] & (df["alps_review_flag"] == "")
print("Alpine candidate set (Matthias):", int(df["in_alpine_candidate_set"].sum()), "| analysis set (without the review flags):", int(df["in_alpine_analysis_set"].sum()))

# Distances to six feeder cities (great-circle, km)
CITIES = {"zurich": (47.3769, 8.5417), "geneva": (46.2044, 6.1432), "milan": (45.4642, 9.1900),
          "turin": (45.0703, 7.6869), "munich": (48.1351, 11.5820), "lyon": (45.7640, 4.8357)}

for city, (clat, clon) in CITIES.items():
    df[f"dist_{city}_km"] = haversine_km(df["latitude"], df["longitude"], clat, clon).round(1)
df["dist_nearest_city_km"] = df[[f"dist_{c}_km" for c in CITIES]].min(axis=1)
print(df[["dist_zurich_km", "dist_nearest_city_km"]].describe().round(1).T)

# Isolation screen (after the corrections): distance to the nearest *other* resort in the table
_lat, _lon = np.radians(df["latitude"].to_numpy()), np.radians(df["longitude"].to_numpy())
_d = 2 * 6371.0088 * np.arcsin(np.sqrt(np.sin((_lat[:, None] - _lat[None]) / 2) ** 2 +
      np.cos(_lat[:, None]) * np.cos(_lat[None]) * np.sin((_lon[:, None] - _lon[None]) / 2) ** 2))
np.fill_diagonal(_d, np.inf)
df["nearest_other_resort_km"] = _d.min(axis=1).round(1)
far = df[(df["in_alpine_analysis_set"]) & (df["nearest_other_resort_km"] > 40)]
print("Analysis-set areas more than 40 km from every other resort (should be none):", far["resort_id"].tolist())

{'geocoders_agree_not_independent': 245, 'verified_access_point': 33, 'two_geocodes_differ_review': 9, 'single_geocode_only': 6}
                               resort_id  latitude  longitude  cache_latitude  cache_longitude  geocode_difference_km
 cima-piazzi-san-colombano-isolaccia-oga 46.472951  10.343436       46.416956        10.285088                    7.7
                     la-plagne-paradiski 45.560071   6.735336       45.504299         6.704912                    6.6
                       laax-flims-falera 46.849618   9.217181       46.806412         9.258127                    5.7
     les-arcs-peisey-vallandry-paradiski 45.575932   6.813361       43.462910         6.478627                  236.5
                    megeve-saint-gervais 45.845128   6.685666       45.856709         6.617932                    5.4
rosskopf-monte-cavallo-sterzing-vipiteno 46.665526  10.824156       46.914452        11.384067                   50.8
          sangiacomo-cardini-monte-alpet 44.2

### 2.3 Prices
Prices stay as scraped (`price_day_raw` in the currency of the listing, `price_day_eur` with the fixed factor 1.05 for CHF; the group has not yet chosen one exchange rate, see `docs/integration_log.md`). Two resorts had no price on the page (Alta Badia, Galtür); Lorenzo entered 74.00 and 56.50 by hand (source not stored in the repository). They are kept **but flagged** `manual_entry`. The silent replacement of other missing prices with the country median is **not** repeated: a price is either scraped or flagged.

In [6]:
df["price_source"] = "scraped"
missing_price = df["price_day_raw"].isna()
manual_prices = lorenzo.drop_duplicates("resort_id").set_index("resort_id")["price_day_eur"]
for rid in df.loc[missing_price, "resort_id"]:
    value = manual_prices.get(rid, np.nan)
    m = df["resort_id"] == rid
    df.loc[m, "price_day_eur"] = value
    df.loc[m, "price_source"] = "manual_entry" if pd.notna(value) else "missing"
    df.loc[m, "currency"] = "EUR"
print(df["price_source"].value_counts().to_dict())
print(df.loc[df["price_source"] != "scraped", ["resort_name", "country", "price_day_eur", "price_source"]])

{'scraped': 291, 'manual_entry': 2}
           resort_name country  price_day_eur  price_source
15          Alta Badia      IT           74.0  manual_entry
87  Galtür – Silvapark      AT           56.5  manual_entry


### 2.4 Variables that need a re-run in the Codespace
- **Slope length, ski routes, lifts:** `python src/reparse_skiresort.py` creates `skiresort_reparsed.csv`. If the file exists it is used; otherwise the old (unverified) slope value is kept and flagged.
- **Hotels:** `python src/fetch_hotels_overpass.py` creates `hotels_overpass.csv`. If the file does not exist the column stays empty.

In [7]:
# Slope, ski routes, lifts
df["ski_routes_km"] = np.nan
df["lifts_count"] = np.nan
df["slope_status"] = "unverified_old_parser"
reparsed_file = PROCESSED / "skiresort_reparsed.csv"
if reparsed_file.exists():
    new = pd.read_csv(reparsed_file).drop_duplicates("resort_id").set_index("resort_id")
    found = new["slope_km"].notna()
    for col in ["slope_km", "ski_routes_km", "lifts_count"]:
        values = new.loc[found, col]
        m = df["resort_id"].isin(values.index)
        df.loc[m, col] = df.loc[m, "resort_id"].map(values)
    df.loc[df["resort_id"].isin(new.index[found]), "slope_status"] = "reparsed"
    changed = (df["slope_status"] == "reparsed").sum()
    print(f"Corrected parser used for {changed} of {len(df)} resorts.")
else:
    print("skiresort_reparsed.csv not found: slope_km stays 'unverified_old_parser' for all resorts.")

# Hotels
df["hotels_4_5_star_10km"] = np.nan
df["hotels_status"] = "not_measured"
hotels_file = PROCESSED / "hotels_overpass.csv"
if hotels_file.exists():
    h = pd.read_csv(hotels_file).drop_duplicates("resort_id").set_index("resort_id")
    m = df["resort_id"].isin(h.index)
    df.loc[m, "hotels_4_5_star_10km"] = df.loc[m, "resort_id"].map(h["hotels_4_5_star_10km"])
    df.loc[m, "hotels_status"] = "openstreetmap_overpass"
    print(f"Real hotel counts for {m.sum()} resorts.")
else:
    print("hotels_overpass.csv not found: hotels_4_5_star_10km stays empty.")

print(df["slope_status"].value_counts().to_dict(), df["hotels_status"].value_counts().to_dict())

skiresort_reparsed.csv not found: slope_km stays 'unverified_old_parser' for all resorts.
hotels_overpass.csv not found: hotels_4_5_star_10km stays empty.
{'unverified_old_parser': 293} {'not_measured': 293}


## 3. Write the tables, the audit file and a check

In [8]:
clean_cols = ["resort_id", "resort_name", "country", "country_memberships", "cross_border", "url",
              "elevation_base_m", "elevation_top_m", "slope_km", "ski_routes_km", "slope_status", "lifts_count",
              "rating_stars", "price_day_raw", "currency", "price_day_eur", "price_source",
              "in_alpine_candidate_set", "in_alpine_analysis_set", "scope_decision", "alps_review_flag"]
spatial_cols = ["latitude", "longitude", "coordinate_method", "coordinate_check", "geocode_difference_km", "nearest_other_resort_km"] + [f"dist_{c}_km" for c in CITIES] + \
               ["dist_nearest_city_km", "hotels_4_5_star_10km", "hotels_status"]

df = df.sort_values("resort_id").reset_index(drop=True)
df[clean_cols].to_csv(PROCESSED / "resorts_clean.csv", index=False)
df[clean_cols + spatial_cols].to_csv(PROCESSED / "resorts_features.csv", index=False)

# Audit file: every row that was removed or changed, with the reason
audit = []
for _, r in lorenzo[lorenzo["manually_added"] == True].iterrows():
    audit.append({"item": r["resort_name"], "action": "removed", "reason": "added by hand, all numbers are the national median"})
for name in ["Scuol – Motta Naluns", "Pila – Aosta"]:
    r = lorenzo[lorenzo["resort_name"] == name].iloc[0]
    audit.append({"item": r["resort_name"] + " (" + r["resort_id"] + ")", "action": "replaced",
                  "reason": "hand-typed numbers under the URL of another resort; original scraped row restored"})
for rid in lost["resort_id"]:
    audit.append({"item": rid, "action": "restored", "reason": "real resort dropped by the 75-per-country quota"})
audit.append({"item": "hotels_4_5_star_10km", "action": "cleared", "reason": "values were a formula of slope and price, not counts"})
audit.append({"item": "lifts_count", "action": "cleared", "reason": "artefact: 12 for every Swiss and Italian row"})
audit.append({"item": "snow_cannons", "action": "dropped", "reason": "0 for every resort (parsing failure)"})
audit.append({"item": "slope_km", "action": "flagged", "reason": "ski-routes figure taken instead of slope length; re-parse needed"})
for rid, (_, _, reason) in COORDINATE_FIXES.items():
    audit.append({"item": rid + " coordinates", "action": "corrected", "reason": reason})
for rid, reason in SCOPE_REVIEW.items():
    audit.append({"item": rid + " scope", "action": "flagged", "reason": reason + "; left out of the analysis set"})
for rid, action, reason in VERIFICATION_ACTIONS:
    audit.append({"item": rid + " coordinates (verification)", "action": action, "reason": reason})
audit = pd.DataFrame(audit)
audit.to_csv(PROCESSED / "resorts_repair_audit.csv", index=False)

status = {"slope_status": sorted(df["slope_status"].unique().tolist()), "hotels_status": sorted(df["hotels_status"].unique().tolist())}
(PROCESSED / "data_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
print(len(audit), "audit rows;", status)

49 audit rows; {'slope_status': ['unverified_old_parser'], 'hotels_status': ['not_measured']}


In [9]:
# Final checks (the notebook stops with an error if one fails)
out = pd.read_csv(PROCESSED / "resorts_features.csv")
assert len(out) == 293 and out["resort_id"].is_unique
assert set(out["resort_id"]) == set(roster["resort_id"]), "must match the roster 1:1"
assert out[["latitude", "longitude", "dist_nearest_city_km"]].notna().all().all()
assert out["country"].isin(["AT", "CH", "FR", "IT"]).all()
assert (out["elevation_base_m"] < out["elevation_top_m"]).all()
assert out["price_day_eur"].notna().all()
assert not out["resort_name"].str.contains("\u200b").any()
assert len(far) == 0, "isolated Alpine resorts: check the coordinates"
assert out["in_alpine_analysis_set"].sum() == out["in_alpine_candidate_set"].sum() - len(set(SCOPE_REVIEW) & set(out.loc[out["in_alpine_candidate_set"], "resort_id"]))
assert out.loc[out["resort_id"].str.startswith("nara-leontica"), "dist_zurich_km"].iloc[0] > 50
print("All checks passed.")
print(out["country"].value_counts().to_dict())
integration_summary = pd.DataFrame([{
    "step": "scraped table -> one row per URL", "rows_in": len(scraped), "rows_out": len(out),
    "removed": len(scraped) - len(out), "note": "7 repeated URLs (5 cross-border areas, 2 repeated rows in one country)"}])
integration_summary

All checks passed.
{'AT': 75, 'IT': 74, 'CH': 73, 'FR': 71}


,step,rows_in,rows_out,removed,note
0,scraped table -> one row per URL,300,293,7,"7 repeated URLs (5 cross-border areas, 2 repeated rows i..."
